In [5]:
# Install required packages
#!pip install nba_api
#!pip install kagglehub


#Download injury data
#import kagglehub

## Download latest version
#path = kagglehub.dataset_download("loganlauton/nba-injury-stats-1951-2023")

#print("Path to dataset files:", path)

import shutil
import os

project_data_path = "./data"

# Create data folder if it doesn't exist
os.makedirs(project_data_path, exist_ok=True)

# Copy all files from downloaded folder
for file in os.listdir(path):
    shutil.copy(os.path.join(path, file), project_data_path)

print("Dataset copied to:", project_data_path)

NameError: name 'path' is not defined

# Importing and wrangling injury data

In [6]:
import pandas as pd
import numpy as np

injuries = pd.read_csv("data/NBA_Injury_Stats.csv")

injuries['Date'] = pd.to_datetime(injuries['Date'])

# Use Dates '2017-10-01' to '2024-06-30' too capture full seasons from 2017 too 2024
injuries_recent = injuries[
    (injuries['Date'] >= '2022-10-01') &
    (injuries['Date'] <= '2024-06-30')
].copy()

injuries_recent = injuries_recent.drop(columns=['Unnamed: 0'])

print(injuries_recent.head())
print("Rows remaining:", len(injuries_recent))

            Date       Team Acquired        Relinquished  \
35942 2022-10-12    Pistons      NaN   Marvin Bagley III   
35943 2022-10-13    Blazers      NaN      Gary Payton II   
35944 2022-10-16     Lakers      NaN     Dennis Schröder   
35945 2022-10-17  Grizzlies      NaN   Jaren Jackson Jr.   
35946 2022-10-18     Lakers      NaN      Troy Brown Jr.   

                                                   Notes  
35942       placed on IL with sprained MCL in right knee  
35943  placed on IL with surgery to repair core muscl...  
35944                     placed on IL with thumb injury  
35945  placed on IL with recovering from surgery on r...  
35946                      placed on IL with back injury  
Rows remaining: 1725


In [7]:
# Get all unique note strings
unique_notes = injuries_recent['Notes'].unique()
print(unique_notes)
print(len(unique_notes))

['placed on IL with sprained MCL in right knee'
 'placed on IL with surgery to repair core muscle injury'
 'placed on IL with thumb injury'
 'placed on IL with recovering from surgery on right foot to repair stress fracture'
 'placed on IL with back injury' 'placed on IL with hip injury'
 'placed on IL with left knee injury'
 'placed on IL recovering from torn ACL in left knee'
 'placed on IL with surgery on left knee'
 'placed on IL with left thigh injury' 'placed on IL with knee injury'
 'placed on IL with foot injury'
 'placed on IL with fractured left big toe'
 'placed on IL with ankle injury' 'placed on IL with sprained left ankle'
 'placed on IL with surgery on right knee'
 'placed on IL with left hamstring injury'
 'placed on IL with left hip injury' 'placed on IL with concussion'
 'placed on IL with strained right calf' 'activated from IL'
 'placed on IL' 'placed on IL with sprained right ankle'
 'placed on IL with laceration in left eye'
 'placed on IL with back spasms' 'place

In [8]:
# Now filter injuries from other reasons for missed time from notes
conditions = [
    injuries_recent["Notes"].str.contains("ACL", case=False, na=False),
    injuries_recent["Notes"].str.contains("Achilles", case=False, na=False),
    injuries_recent["Notes"].str.contains("knee", case=False, na=False),
    injuries_recent["Notes"].str.contains("ankle", case=False, na=False),
    injuries_recent["Notes"].str.contains("hamstring", case=False, na=False),
    injuries_recent["Notes"].str.contains("calf", case=False, na=False)
]

choices = ["ACL", "Achilles", "Knee", "Ankle", "Hamstring", "Calf"]

injuries_recent["injury_type"] = np.select(conditions, choices, default="Other")

# Filter out "Other" injury types
injuries_filtered = injuries_recent[injuries_recent.injury_type != "Other"]

injuries_filtered.rename(columns={'Relinquished': 'player_name'}, inplace=True)

# split into two columns on '/'
injuries_filtered[['name_1', 'name_2']] = injuries_filtered['player_name'].str.split('/', n=1, expand=True)

# clean whitespace
injuries_filtered['name_1'] = injuries_filtered['name_1'].str.strip()
injuries_filtered['name_2'] = injuries_filtered['name_2'].str.strip()

# Cleaning you injury df
injuries_filtered["name_1"] = injuries_filtered["name_1"].str.strip().str.lower()
injuries_filtered["name_2"] = injuries_filtered["name_2"].str.strip().str.lower()

# Show first 10 rows
injuries_filtered.head(3)

C:\Users\antho\AppData\Local\Temp\ipykernel_17984\1955913624.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  injuries_filtered.rename(columns={'Relinquished': 'player_name'}, inplace=True)
C:\Users\antho\AppData\Local\Temp\ipykernel_17984\1955913624.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  injuries_filtered[['name_1', 'name_2']] = injuries_filtered['player_name'].str.split('/', n=1, expand=True)
C:\Users\antho\AppData\Local\Temp\ipykernel_17984\1955913624.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

,Date,Team,Acquired,player_name,Notes,injury_type,name_1,name_2
35942,2022-10-12,Pistons,NaN,Marvin Bagley III,placed on IL with sprained MCL in right knee,Knee,marvin bagley iii,None
35948,2022-10-19,Bulls,NaN,Zach LaVine,placed on IL with left knee injury,Knee,zach lavine,None
35949,2022-10-19,Cavaliers,NaN,Ricky Rubio,placed on IL recovering from torn ACL in left ...,ACL,ricky rubio,None


# Extracting from NBA API

# Combining GAME_ID and PLAYER_ID and SEASON_ID with injury data 

In [9]:
from nba_api.stats.endpoints import commonallplayers


# Fetch all players for a given season
players = commonallplayers.CommonAllPlayers(is_only_current_season=0, league_id='00').get_data_frames()[0]

columns_to_keep = ['PERSON_ID', 'DISPLAY_FIRST_LAST', 'ROSTERSTATUS', 'TEAM_ID','TEAM_CITY','FROM_YEAR','TO_YEAR']

players = players[columns_to_keep]

players = players.rename(columns={"DISPLAY_FIRST_LAST": "name"}).copy()

# clean names
players["name"] = players["name"].str.strip().str.lower()

players.head(5)

,PERSON_ID,name,ROSTERSTATUS,TEAM_ID,TEAM_CITY,FROM_YEAR,TO_YEAR
0,76001,alaa abdelnaby,0,0,,1990,1994
1,76002,zaid abdul-aziz,0,0,,1968,1977
2,76003,kareem abdul-jabbar,0,0,,1969,1988
3,51,mahmoud abdul-rauf,0,0,,1990,2000
4,1505,tariq abdul-wahad,0,0,,1997,2003


In [10]:
import pandas as pd
import numpy as np
import re

import re
import unicodedata

# MERGE PLAYER-ID TO INJURIES

def clean_name(x):
    if pd.isna(x):
        return x
    
    x = str(x).lower()
    
    # remove accents (Jokić → Jokic)
    x = unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8')
    
    # remove anything in parentheses
    x = re.sub(r"\(.*?\)", "", x)
    
    # remove suffixes
    x = re.sub(r"\b(jr\.?|sr\.?|ii|iii|iv)\b", "", x)
    
    # remove punctuation (., etc)
    x = re.sub(r"[^\w\s]", "", x)
    
    # normalize whitespace
    x = re.sub(r"\s+", " ", x).strip()
    
    return x

# df1 (players)
players["name"] = players["name"].apply(clean_name)

# df2 (injury data)
injuries_filtered["name_1"] = injuries_filtered["name_1"].apply(clean_name)
injuries_filtered["name_2"] = injuries_filtered["name_2"].apply(clean_name)

merged = injuries_filtered.merge(
    players[["PERSON_ID", "name"]],
    left_on="name_1",
    right_on="name",
    how="left"
).rename(columns={"PERSON_ID": "PERSON_ID_1"})

merged = merged.merge(
    players[["PERSON_ID", "name"]],
    left_on="name_2",
    right_on="name",
    how="left"
).rename(columns={"PERSON_ID": "PERSON_ID_2"})

merged["PERSON_ID"] = merged["PERSON_ID_1"].combine_first(merged["PERSON_ID_2"])

#clean up

merged = merged.drop(columns=["name_1", "name_2", "PERSON_ID_1", "name_x", "PERSON_ID_2", "name_y"])
merged = merged.dropna(subset=['PERSON_ID'])

merged.head(3)

C:\Users\antho\AppData\Local\Temp\ipykernel_17984\173816285.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  injuries_filtered["name_1"] = injuries_filtered["name_1"].apply(clean_name)
C:\Users\antho\AppData\Local\Temp\ipykernel_17984\173816285.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  injuries_filtered["name_2"] = injuries_filtered["name_2"].apply(clean_name)


,Date,Team,Acquired,player_name,Notes,injury_type,PERSON_ID
0,2022-10-12,Pistons,NaN,Marvin Bagley III,placed on IL with sprained MCL in right knee,Knee,1628963.0
1,2022-10-19,Bulls,NaN,Zach LaVine,placed on IL with left knee injury,Knee,203897.0
2,2022-10-19,Cavaliers,NaN,Ricky Rubio,placed on IL recovering from torn ACL in left ...,ACL,201937.0


In [81]:
len(merged)

379

In [11]:
from nba_api.stats.endpoints import playergamelog, commonallplayers
import pandas as pd
import time

player_ids = merged['PERSON_ID'].dropna().astype(int).unique()
seasons = ['2022-23','2023-24','2024-25']

# File to save incrementally
csv_file = "gamelogs.csv"

# Load existing CSV if it exists
if os.path.exists(csv_file):
    full_gamelog = pd.read_csv(csv_file)
    fetched_players = full_gamelog['PLAYER_ID'].unique()
else:
    full_gamelog = pd.DataFrame()
    fetched_players = []

for pid in player_ids:
    if pid in fetched_players:
        print(f"Skipping already fetched player {pid}")
        continue

    player_logs = []

    for season in seasons:
        for attempt in range(3):  # retry logic
            try:
                gamelog = playergamelog.PlayerGameLog(
                    player_id=pid,
                    season=season,
                    season_type_all_star='Regular Season',
                    timeout=60
                )
                df = gamelog.get_data_frames()[0]
                if df.empty:
                    break
                df.rename(columns=lambda x: x.upper(), inplace=True)
                columns_to_keep = ['SEASON_ID','PLAYER_ID','GAME_ID','GAME_DATE','MATCHUP','MIN']
                existing_cols = [col for col in columns_to_keep if col in df.columns]
                df = df[existing_cols]
                player_logs.append(df)
                break  # success
            except Exception as e:
                print(f"Attempt {attempt+1} failed for player {pid} season {season}: {e}")
                time.sleep(1)
                if attempt == 2:
                    print(f"Skipping player {pid} season {season} after 3 attempts")
        time.sleep(0.6)  # avoid API throttle

    if player_logs:
        player_df = pd.concat(player_logs, ignore_index=True)
        full_gamelog = pd.concat([full_gamelog, player_df], ignore_index=True)
        # Save to CSV after each player
        full_gamelog.to_csv(csv_file, index=False)
        print(f"Saved logs for player {pid}, total rows: {len(full_gamelog)}")

Skipping already fetched player 1628963
Skipping already fetched player 203897
Skipping already fetched player 201937
Skipping already fetched player 201980
Skipping already fetched player 203506
Skipping already fetched player 203914
Skipping already fetched player 202722
Skipping already fetched player 1628373
Skipping already fetched player 203552
Skipping already fetched player 1628464
Skipping already fetched player 1628449
Skipping already fetched player 203920
Skipping already fetched player 203490
Skipping already fetched player 1630256
Skipping already fetched player 204060
Skipping already fetched player 1626192
Skipping already fetched player 1630591
Skipping already fetched player 1626179
Skipping already fetched player 1630567
Skipping already fetched player 1630581
Skipping already fetched player 1628978
Skipping already fetched player 202695
Skipping already fetched player 1630529
Skipping already fetched player 201566
Skipping already fetched player 1630170
Skipping alr

KeyboardInterrupt: 

In [83]:
merged.head(5)

,Date,Team,Acquired,player_name,Notes,injury_type,PERSON_ID
0,2022-10-12,Pistons,NaN,Marvin Bagley III,placed on IL with sprained MCL in right knee,Knee,1628963.0
1,2022-10-19,Bulls,NaN,Zach LaVine,placed on IL with left knee injury,Knee,203897.0
2,2022-10-19,Cavaliers,NaN,Ricky Rubio,placed on IL recovering from torn ACL in left ...,ACL,201937.0
3,2022-10-19,Grizzlies,NaN,Danny Green,placed on IL with surgery on left knee,Knee,201980.0
4,2022-10-19,Heat,NaN,Victor Oladipo,placed on IL with knee injury,Knee,203506.0


In [12]:
import pandas as pd
import numpy as np

# -----------------------------
# 1️⃣ LOAD DATA
# -----------------------------
gamelog_df = pd.read_csv('gamelogs.csv')

# -----------------------------
# 2️⃣ CLEAN INJURY DATA
# -----------------------------
merged = merged.dropna(subset=['PERSON_ID'])

merged['PERSON_ID'] = merged['PERSON_ID'].astype(int)
merged['Date'] = pd.to_datetime(merged['Date'])

merged = merged.sort_values(['PERSON_ID', 'Date']).reset_index(drop=True)

# -----------------------------
# 3️⃣ CLEAN GAME LOG DATA
# -----------------------------
gamelog_df['PLAYER_ID'] = gamelog_df['PLAYER_ID'].astype(int)
gamelog_df['GAME_DATE'] = pd.to_datetime(gamelog_df['GAME_DATE'])

gamelog_df = gamelog_df.sort_values(['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

# -----------------------------
# 4️⃣ FIND LAST GAME BEFORE EACH INJURY
# -----------------------------
result = []

for pid, injury_group in merged.groupby('PERSON_ID'):
    
    player_games = gamelog_df[gamelog_df['PLAYER_ID'] == pid]
    
    if player_games.empty:
        continue
    
    injury_group = injury_group.sort_values('Date')
    player_games = player_games.sort_values('GAME_DATE')
    
    merged_part = pd.merge_asof(
        injury_group,
        player_games[['PLAYER_ID','GAME_DATE','GAME_ID','SEASON_ID']],
        left_on='Date',
        right_on='GAME_DATE',
        direction='backward'
    )
    
    result.append(merged_part)

# Combine
injury_with_games = pd.concat(result).reset_index(drop=True)

# -----------------------------
# 5️⃣ CLEAN + FEATURE
# -----------------------------
injury_with_games['days_since_last_game'] = (
    injury_with_games['Date'] - injury_with_games['GAME_DATE']
).dt.days

# Optional: filter unrealistic matches
injury_with_games = injury_with_games[
    injury_with_games['days_since_last_game'] <= 14
]

# -----------------------------
# 6️⃣ CREATE INJURY LABEL DATASET
# -----------------------------
# Keep only what we need to merge into gamelog
injury_labels = injury_with_games[[
    'PERSON_ID',
    'GAME_ID',
    'Date',
    'injury_type',
    'player_name'
]].copy()

# Add label
injury_labels['injury'] = 1

# Rename for merge
injury_labels = injury_labels.rename(columns={'PERSON_ID': 'PLAYER_ID'})

# -----------------------------
# 7️⃣ MERGE INTO GAMELOG DATA
# -----------------------------
# Start with all games
gamelog_df['injury'] = 0

# Merge injury labels onto gamelog
gamelog_df = gamelog_df.merge(
    injury_labels[['PLAYER_ID','GAME_ID','injury','injury_type']],
    on=['PLAYER_ID','GAME_ID'],
    how='left',
    suffixes=('','_y')
)

# Fill injury column
gamelog_df['injury'] = gamelog_df['injury_y'].fillna(0)

# Optional: keep injury_type only where injury = 1
gamelog_df['injury_type'] = gamelog_df['injury_type'].fillna('None')

# Drop temp column
gamelog_df = gamelog_df.drop(columns=['injury_y'])

# -----------------------------
# 8️⃣ VALIDATION
# -----------------------------
print("\n--- VALIDATION ---")

# How many injury games
print("Total injury games:", gamelog_df['injury'].sum())

# % injury games
print("Injury rate:", gamelog_df['injury'].mean())

# Check duplicates (should be none)
dup_check = gamelog_df.duplicated(subset=['PLAYER_ID','GAME_ID']).sum()
print("Duplicate game rows:", dup_check)

# Sample
print("\nSample labeled data:")
print(gamelog_df[['PLAYER_ID','GAME_ID','GAME_DATE','injury','injury_type']].head(10))

# -----------------------------
# 9️⃣ SAVE
# -----------------------------
gamelog_df.to_csv('gamelog_with_injuries.csv', index=False)

print("\n✅ Gamelog dataset with injuries ready!")


--- VALIDATION ---
Total injury games: 308.0
Injury rate: 0.011896944648306231
Duplicate game rows: 2

Sample labeled data:
   PLAYER_ID   GAME_ID  GAME_DATE  injury injury_type
0       2544  22200002 2022-10-18     0.0        None
1       2544  22200016 2022-10-20     0.0        None
2       2544  22200037 2022-10-23     0.0        None
3       2544  22200064 2022-10-26     0.0        None
4       2544  22200076 2022-10-28     0.0        None
5       2544  22200095 2022-10-30     0.0        None
6       2544  22200117 2022-11-02     0.0        None
7       2544  22200131 2022-11-04     0.0        None
8       2544  22200140 2022-11-06     0.0        None
9       2544  22200170 2022-11-09     0.0        None

✅ Gamelog dataset with injuries ready!


# Add predictors too Game level injury data

In [ ]:
import time
import pandas as pd
from nba_api.stats.endpoints import BoxScorePlayerTrackV3
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import requests
import random

# =====================
# 1️⃣ Fetch single game for specific players
# =====================
def fetch_game_tracking(game_id, player_ids, retries=3, base_delay=5):
    """Fetch minimal tracking stats for a single game for specified players."""
    for attempt in range(retries):
        try:
            time.sleep(random.uniform(0.5, 1.5))  # jitter to avoid throttling
            boxscore = BoxScorePlayerTrackV3(game_id=game_id)
            df = boxscore.get_data_frames()[0]
            df = df[['gameId', 'personId', 'minutes', 'speed', 'distance']]
            df = df[df['personId'].isin(player_ids)]
            return df
        except (requests.exceptions.RequestException, ConnectionResetError) as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0,2)
            print(f"Attempt {attempt+1} failed for {game_id}: {e}. Retrying in {wait:.1f}s.")
            time.sleep(wait)
    print(f"Failed to fetch game {game_id} after {retries} retries.")
    return pd.DataFrame(columns=['gameId', 'personId', 'minutes', 'speed', 'distance'])


# =====================
# 2️⃣ Fetch games in chunks with incremental save
# =====================
def fetch_all_tracking_chunks(injury_df, save_path='tracking_partial.csv',
                              chunk_size=100, max_workers=1, save_every=10,
                              retries=3, delay=5):
    """
    Fetch player tracking stats in smaller chunks for stability.
    Saves partial CSV after every batch and shows total unique games and player-games fetched.
    """
    injury_df['GAME_ID'] = injury_df['GAME_ID'].astype(str).str.zfill(10)
    injury_df['PLAYER_ID'] = injury_df['PLAYER_ID'].astype(int)
    
    all_tracking = []
    
    # Resume from previous partial file
    if os.path.exists(save_path):
        partial_df = pd.read_csv(save_path)
        # Drop duplicates just in case
        partial_df = partial_df.drop_duplicates(subset=['gameId', 'personId'], keep='first')
        completed_games = set(partial_df['gameId'].unique())
        all_tracking.append(partial_df)
        print(f"Resuming from saved file: {len(completed_games)} games, {len(partial_df)} player-games already fetched.")
    else:
        completed_games = set()

    # Filter out already fetched player-games
    already_fetched = set()
    if all_tracking:
        df_concat = pd.concat(all_tracking, ignore_index=True)
        already_fetched = set(tuple(x) for x in df_concat[['gameId','personId']].values)

    # Build the list of unique games to fetch
    unique_games = [gid for gid in injury_df['GAME_ID'].unique()]
    print(f"{len(unique_games)} total games in injury df...")

    # Break into chunks
    game_chunks = [unique_games[i:i+chunk_size] for i in range(0, len(unique_games), chunk_size)]
    
    for chunk_idx, chunk in enumerate(game_chunks, start=1):
        print(f"\n=== Fetching chunk {chunk_idx} / {len(game_chunks)}: {len(chunk)} games ===")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_game = {}
            for gid in chunk:
                player_ids = injury_df[injury_df['GAME_ID'] == gid]['PLAYER_ID'].tolist()
                # Filter out already fetched player-games
                player_ids = [pid for pid in player_ids if (gid, pid) not in already_fetched]
                if player_ids:
                    future = executor.submit(fetch_game_tracking, gid, player_ids, retries, delay)
                    future_to_game[future] = gid
            
            for future in as_completed(future_to_game):
                df_game = future.result()
                if not df_game.empty:
                    all_tracking.append(df_game)
                    # Update sets to avoid redownloading
                    completed_games.add(df_game['gameId'].iloc[0])
                    for pair in df_game[['gameId','personId']].itertuples(index=False):
                        already_fetched.add((pair.gameId, pair.personId))

        # Save after each chunk
        partial_df = pd.concat(all_tracking, ignore_index=True)
        partial_df.to_csv(save_path, index=False)

        total_player_games = len(partial_df[['gameId','personId']].drop_duplicates())
        total_games_fetched = len(partial_df['gameId'].drop_duplicates())
        total_player_games_expected = len(injury_df[['GAME_ID','PLAYER_ID']].drop_duplicates())
        print(f"Saved partial CSV after chunk {chunk_idx}: {len(partial_df)} rows, {total_games_fetched} unique games, {total_player_games}/{total_player_games_expected} player-games fetched.")
    
    tracking_df = pd.concat(all_tracking, ignore_index=True)
    tracking_df.to_csv(save_path, index=False)
    print("Saved final tracking data.")
    
    # Return tracking df and list of failed games
    fetched_games = set(tracking_df['gameId'].unique())
    failed_games = [gid for gid in unique_games if gid not in fetched_games]
    return tracking_df, failed_games


# =====================
# 3️⃣ Retry failed games separately
# =====================
def retry_failed_games(failed_games, injury_df, save_path='tracking_partial.csv',
                       retries=5, delay=15):
    all_tracking = []
    print(f"Retrying {len(failed_games)} failed games serially...")
    for gid in failed_games:
        player_ids = injury_df[injury_df['GAME_ID'] == gid]['PLAYER_ID'].tolist()
        df = fetch_game_tracking(gid, player_ids, retries=retries, base_delay=delay)
        if not df.empty:
            all_tracking.append(df)
        if all_tracking:
            partial_df = pd.read_csv(save_path) if os.path.exists(save_path) else pd.DataFrame()
            partial_df = pd.concat([partial_df] + all_tracking, ignore_index=True).drop_duplicates(subset=['gameId','personId'])
            partial_df.to_csv(save_path, index=False)
    return pd.concat(all_tracking, ignore_index=True) if all_tracking else pd.DataFrame()


# =====================
# 4️⃣ Merge tracking with injury data
# =====================
def merge_tracking_with_injury(injury_df, tracking_df):
    merged_full = injury_df.merge(
        tracking_df,
        left_on=['PLAYER_ID', 'GAME_ID'],
        right_on=['personId', 'gameId'],
        how='left'
    )
    merged_full['minutes'] = pd.to_numeric(merged_full['minutes'], errors='coerce')
    merged_full['distance'] = pd.to_numeric(merged_full['distance'], errors='coerce')
    
    merged_full = merged_full.sort_values(['PLAYER_ID', 'GAME_DATE'])
    merged_full['distance_rolling'] = merged_full.groupby('PLAYER_ID')['distance']\
                                                .rolling(3, min_periods=1).mean().reset_index(0, drop=True)
    merged_full['minutes_rolling'] = merged_full.groupby('PLAYER_ID')['minutes']\
                                                .rolling(3, min_periods=1).mean().reset_index(0, drop=True)
    return merged_full


# =====================
# 5️⃣ Usage example
# =====================
merged_injury_df = pd.read_csv('gamelog_with_injuries.csv')

# Main fetch in chunks
tracking_df, failed_games = fetch_all_tracking_chunks(
    merged_injury_df,
    save_path='tracking_partial.csv',
    chunk_size=100,
    max_workers=1,  # safer for API
    save_every=10,
    retries=3,
    delay=7
)

# Retry any failed games
if failed_games:
    retry_failed_games(failed_games, merged_injury_df, save_path='tracking_partial.csv',
                       retries=5, delay=15)

# Reload final tracking data
tracking_df = pd.read_csv('tracking_partial.csv')

# Merge with injury dataframe
merged_full_df = merge_tracking_with_injury(merged_injury_df, tracking_df)
merged_full_df.to_csv('gamelog_with_tracking.csv', index=False)
print("Merged dataframe saved as gamelog_with_tracking.csv")

Resuming from saved file: 600 games, 4671 player-games already fetched.
3688 total games in injury df...

=== Fetching chunk 1 / 37: 100 games ===
Saved partial CSV after chunk 1: 5535 rows, 700 unique games, 5535/25887 player-games fetched.

=== Fetching chunk 2 / 37: 100 games ===


In [2]:
import pandas as pd

merged_injury_df = pd.read_csv('gamelog_with_injuries.csv')

# Ensure consistent types
merged_injury_df['GAME_ID'] = merged_injury_df['GAME_ID'].astype(str).str.zfill(10)
merged_injury_df['PLAYER_ID'] = merged_injury_df['PLAYER_ID'].astype(int)

# Count unique player-game combinations
unique_player_games = merged_injury_df[['GAME_ID', 'PLAYER_ID']].drop_duplicates()
total_unique_player_games = len(unique_player_games)

# Count unique games as well
total_unique_games = merged_injury_df['GAME_ID'].nunique()

print(f"Total unique player-games to fetch: {total_unique_player_games}")
print(f"Total unique games to fetch: {total_unique_games}")

Total unique player-games to fetch: 25887
Total unique games to fetch: 3688


In [4]:
import pandas as pd

# Load your partial CSV
csv_file = 'tracking_partial.csv'
df = pd.read_csv(csv_file)

# Total rows
total_rows = len(df)

# Count unique game-player combinations
unique_game_player = df[['gameId', 'personId']].drop_duplicates()
unique_game_player_count = len(unique_game_player)

# Count unique games
unique_games = df['gameId'].drop_duplicates()
unique_games_count = len(unique_games)

# Find duplicates (same player in same game)
duplicates = df.duplicated(subset=['gameId', 'personId'], keep=False)

print(f"Total rows in CSV: {total_rows}")
print(f"Unique game-player combinations: {unique_game_player_count}")
print(f"Unique games: {unique_games_count}")
print(f"Number of duplicated game-player rows: {duplicates.sum()}")

# Optional: show duplicate rows
if duplicates.any():
    print("\nSample duplicated rows:")
    print(df[duplicates].head(20))

Total rows in CSV: 4671
Unique game-player combinations: 4671
Unique games: 600
Number of duplicated game-player rows: 0
